# EV7 Taxi — ถามตอบภาษาไทยแบบไม่ต่อ API (Google Colab)

<a href="https://colab.research.google.com/" target="_blank">เปิดใน Colab</a> → File → Upload notebook → เลือกไฟล์นี้

**ก่อนรัน**
1. ในเครื่อง: `python scripts/taxi/qa/make_colab_bundle.py` → ได้ `D:\Taxi\colab\taxi_qa_data.zip` (ทะเบียนถูกแทนด้วย `EV-xxxx` แล้ว)
2. อัปโหลด zip ไปที่ Google Drive โฟลเดอร์ `MyDrive/taxi_qa/` (**ใช้ Drive ขององค์กร** — ไม่ใช่ Gmail ส่วนตัว)
3. ส่วน 1–7 ใช้ CPU ได้ · ส่วน 8 (fine-tune LLM) ต้องเปลี่ยนเป็น GPU: Runtime → Change runtime type → **T4 GPU**

> 🔒 ห้ามอัปโหลด `plate_map_LOCAL.csv` (ตารางจับคู่ทะเบียนจริง) · พิกัดรถยังอยู่ในชุดข้อมูล ถือเป็นข้อมูลอ่อนไหว — ขออนุมัติก่อนใช้ cloud ภายนอก

---

ถามเป็นภาษาไทย เช่น *"ตอนนี้รถคันไหนอยู่ภูเกตบ้าง"* → ได้คำตอบจากข้อมูลจริง **โดยไม่ส่งข้อมูลออกนอกเครื่อง**

## แนวคิด: แยกคำถามเป็นชิ้น แทนการให้ LLM คิดทั้งหมด

```
คำถาม ─► ① ดึงค่า (slot)      ทะเบียน · สถานที่ (ทนพิมพ์ผิด) · เวลา      ← กฎ + fuzzy match
       ─► ② จัดประเภทคำถาม   intent: หารถในพื้นที่ / สถานะรถ / ...    ← embedder + classifier (เทรนเอง)
       ─► ③ query ข้อมูลจริง  pandas บน points_clean.parquet            ← โค้ดที่เราเขียน · ตอบถูกทุกแถว
       ─► ④ สร้างคำตอบไทย      template                                ← ไม่แต่งเรื่อง
```

ส่วน ② ใช้ **backbone ภาษา** ตัวเดียวกับโมเดลจัดหมวดสินค้า ITEC (`paraphrase-multilingual-MiniLM-L12-v2` ที่ cache ไว้แล้ว) + เทรน classifier ด้านบนเอง — แนวเดียวกับ `scripts/itec/category`

ท้าย notebook (ส่วน 8) มีแบบ **ใช้ LLM โอเพนซอร์สรันเอง + fine-tune (LoRA)** สำหรับเมื่อคำถามหลากหลายเกินกว่า intent ที่กำหนด — ต้องมี GPU

> ⚠️ ผลลัพธ์มี **ทะเบียนรถและตำแหน่ง** (ข้อมูลส่วนบุคคล) · **ล้าง output (Clear All Outputs) ก่อน commit / ส่งต่อ**
> ข้อมูลเป็น batch (30 ก.ย. – 1 ต.ค. 2026) — คำว่า "ตอนนี้" = เวลาล่าสุดของข้อมูล ไม่ใช่เวลาจริง

แบบระบบรวม → `COM7-Knowledge-Base/06_Project/EV7 Taxi - Q&A Assistant Design.md`

## 0 · ตั้งค่า Colab — เชื่อม Drive และแตกไฟล์ข้อมูล

**Colab คืออะไร** — Jupyter notebook ที่รันบนเครื่องของ Google ผ่านเบราว์เซอร์ ไม่ต้องติดตั้งอะไรในเครื่องเรา · ทุกครั้งที่เปิดใหม่ เครื่องจะว่างเปล่า (ไฟล์ที่สร้างหายเมื่อปิด) จึงเก็บข้อมูลไว้ใน Google Drive

**วิธีรัน cell** — คลิกที่ cell แล้วกด `Shift + Enter` (หรือปุ่ม ▶ ซ้ายของ cell) · รันจากบนลงล่างทีละ cell · cell ที่กำลังรันจะมีวงหมุน

**ขั้นตอนครั้งแรก**
1. อัปโหลด `taxi_qa_data.zip` ไปไว้ใน Google Drive → สร้างโฟลเดอร์ชื่อ `taxi_qa` ใน My Drive → ลากไฟล์ใส่
2. รัน cell ข้างล่าง → จะมีหน้าต่างถาม **"Permit this notebook to access your Google Drive files?"** → กด **Connect to Google Drive** → เลือกบัญชี (ขององค์กร) → Allow
3. ผลที่ถูก: บรรทัดสุดท้ายแสดง `['dim_province.csv', 'dist_pip.geojson', 'ev_chargers_osm.json', 'points.parquet']`

| ถ้าเจอ | แปลว่า | แก้ |
|---|---|---|
| `FileNotFoundError ... taxi_qa_data.zip` | zip ไม่ได้อยู่ใน `MyDrive/taxi_qa/` | ตรวจชื่อโฟลเดอร์/ไฟล์ (ตัวพิมพ์เล็ก-ใหญ่มีผล) |
| ไม่มีหน้าต่างขออนุญาต | browser บล็อก popup | อนุญาต popup ของ colab.research.google.com |

In [ ]:
import sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/taxi_qa")          # โฟลเดอร์ที่วาง taxi_qa_data.zip
    DATA = Path("/content/taxi_qa")                          # แตกไฟล์มาไว้บนเครื่อง Colab (อ่านเร็วกว่า Drive)
    DATA.mkdir(exist_ok=True)
    zipfile.ZipFile(DRIVE / "taxi_qa_data.zip").extractall(DATA)
    # ไม่ใช้ Drive: from google.colab import files; files.upload() แล้วแตก zip ที่อัปโหลดมาเอง
else:
    DATA = Path(r"D:\Taxi\colab\taxi_qa")                  # ทดสอบในเครื่อง (ชุดเดียวกับที่อัปโหลด)
print("ข้อมูลอยู่ที่", DATA, "·", sorted(p.name for p in DATA.iterdir()))

## 1 · โหลดข้อมูล และเตรียมตาราง `pos`

**ข้อมูลคืออะไร** — `points.parquet` คือจุด GPS ของรถแท็กซี่ 1.17 ล้านจุด (ไฟล์ parquet = ตารางแบบบีบอัด อ่านด้วย pandas ได้เร็ว)
แต่ละแถว = รถ 1 คัน ณ เวลาหนึ่ง

| คอลัมน์ | ความหมาย |
|---|---|
| `car_id` / `plate` | รหัสรถสมมติ `EV-0001` (แทนทะเบียนจริงแล้ว) · 1 รหัส = 1 คัน (มาจาก VIN) |
| `lat` `lon` | พิกัด (ละติจูด ลองจิจูด) |
| `speed` `direction` | ความเร็ว กม./ชม. · ทิศหัวรถเป็นองศา (0 = เหนือ, 90 = ตะวันออก) |
| `meter` | `F` = มีผู้โดยสาร · `E` = ว่าง |
| `signal` | `A` = GPS ปกติ · `V` = ไม่มีสัญญาณ |
| `dist` | ลำดับอำเภอที่จุดนั้นอยู่ → cell นี้แปลงเป็นชื่อ/รหัสอำเภอและจังหวัด |

cell นี้เพิ่มคอลัมน์ `status` ด้วยกฎง่าย ๆ: ไม่มีสัญญาณ → มีผู้โดยสาร → วิ่ง (เร็วกว่า 5 กม./ชม.) → นอกนั้นคือจอด
`LATEST` = เวลาล่าสุดในข้อมูล — ระบบใช้แทนคำว่า "ตอนนี้" เพราะข้อมูลเป็นชุดเก่า (ไม่ใช่ realtime)

In [ ]:
import json, re, math, random, difflib, time
from pathlib import Path
import numpy as np
import pandas as pd

OUT = MASTER = DATA                                        # ทุกไฟล์อยู่ในโฟลเดอร์เดียว (จาก zip)

pts = pd.read_parquet(OUT / "points.parquet",
                      columns=["car_id", "plate", "meter", "signal", "lat", "lon", "speed", "direction", "gps_time", "dist"])
# car_id ในชุด Colab = รหัสสมมติต่อ VIN (1 คัน = 1 รหัส) ทำโดย make_colab_bundle.py
dist = pd.DataFrame([f["properties"] for f in json.load(open(OUT / "dist_pip.geojson", encoding="utf-8"))["features"]])
# คอลัมน์ dist ใน points = ลำดับ feature ใน dist_pip.geojson (สร้างโดย build_dashboard.assign_area)
pos = pts.assign(
    dist_code=dist.pc.str[2:].values[pts["dist"]],          # TH8301 → 8301
    prov_code=dist.ppc.str[2:].values[pts["dist"]],          # TH83 → 83
    dist_th=dist.th.values[pts["dist"]],
    prov_th=dist.pth.values[pts["dist"]],
    plate_key=pts.plate.str.replace(r"[\s-]", "", regex=True),   # ทอ-4141 / ทอ 4141 → ทอ4141
)
MOVING_KMH = 5
pos["status"] = np.select(
    [pos.signal.eq("V"), pos.meter.eq("F"), pos.speed.ge(MOVING_KMH)],
    ["ไม่มีสัญญาณ", "มีผู้โดยสาร", "ว่าง·วิ่ง"], default="จอด")
pos = pos.sort_values(["car_id", "gps_time"]).reset_index(drop=True)
LATEST = pos.gps_time.max()
print(f"{len(pos):,} จุด · {pos.car_id.nunique():,} คัน · ข้อมูลล่าสุด {LATEST}")
pos.head(3)

## 2 · รายชื่อสถานที่ (gazetteer)

**ทำไมต้องมี** — เวลาผู้ใช้พิมพ์ "ภูเก็ต" ระบบต้องรู้ว่าคือจังหวัดรหัส 83 · cell นี้รวมชื่อทุกจังหวัด (77) และอำเภอ (928) ทั้งไทย อังกฤษ และแบบมีคำนำหน้า ("จังหวัด…", "อำเภอ…", "เขต…" สำหรับกรุงเทพฯ) + ชื่อเล่นที่คนใช้บ่อย (กทม · โคราช)

อยากเพิ่มชื่อเล่น → แก้ `ALIAS` เช่น `"หาดใหญ่": ...` (ใส่รหัสจังหวัด 2 หลัก)

In [ ]:
prov = dist[["ppc", "pth", "pen"]].drop_duplicates("ppc")
gaz = []                                                     # (ชื่อที่ใช้จับ, ระดับ, รหัส, ชื่อเต็มที่แสดง)
for r in prov.itertuples():
    for n in {r.pth, "จังหวัด" + r.pth, r.pen.lower()}:
        gaz.append((n, "prov", r.ppc[2:], r.pth))
for r in dist.itertuples():
    pre = "เขต" if r.ppc == "TH10" else "อำเภอ"             # "เขต" ใช้เฉพาะกรุงเทพฯ
    for n in {r.th, pre + r.th, r.en.lower()}:
        gaz.append((n, "dist", r.pc[2:], f"{r.th} ({r.pth})"))
ALIAS = {"กทม": "10", "กรุงเทพ": "10", "บางกอก": "10", "bkk": "10", "โคราช": "30", "อยุธยา": "14"}
name_of = dict(zip(prov.ppc.str[2:], prov.pth))
gaz += [(a, "prov", c, name_of[c]) for a, c in ALIAS.items()]
gaz = pd.DataFrame(gaz, columns=["name", "level", "code", "label"]).drop_duplicates("name")

# จุดกึ่งกลางจังหวัด (จาก D:\Master) ไว้หาสถานีชาร์จใกล้จังหวัด
cent = pd.read_csv(MASTER / "dim_province.csv", dtype={"province_code": str}, encoding="utf-8-sig") \
         .set_index("province_code")[["centroid_lat", "centroid_lon"]]
print(len(gaz), "ชื่อ ·", gaz.level.value_counts().to_dict())

## 3 · ดึงค่าจากคำถาม (slot) — ทะเบียน · เวลา · สถานที่

**slot** = ค่าที่ต้องรู้ก่อนไปค้นข้อมูล — ถามว่า "EV-0042 อยู่ไหน" ต้องรู้ว่าทะเบียนคือ EV-0042

| slot | วิธีหา | ตัวอย่าง |
|---|---|---|
| ทะเบียน | **regex** (รูปแบบตัวอักษร): ตัวอักษร 1–3 ตัว + ตัวเลข 3–4 หลัก มีขีด/เว้นวรรคหรือไม่ก็ได้ | `EV-0042` `EV 0042` `ทอ-4141` |
| เวลา | คำสำคัญ: ตอนนี้/ล่าสุด · วันนี้ · เมื่อวาน · `14:30` · `9 โมง` | "เมื่อวาน" → ช่วงวันก่อนหน้าวันล่าสุดของข้อมูล |
| สถานที่ | **fuzzy match**: เทียบคำถามกับทุกชื่อใน gazetteer ให้คะแนนความเหมือน 0–1 | "ภูเกต" กับ "ภูเก็ต" ได้ 0.91 → ผ่านเกณฑ์ 0.85 |

ชื่อสั้นกว่า 4 ตัวอักษรต้องตรงเป๊ะ — ไม่งั้นคำทั่วไปอย่าง "ไปไหนมา" จะถูกจับเป็นอำเภอ "สนม"

**ลองเอง:** แก้ประโยคใน list ท้าย cell แล้วรันใหม่ ดูว่าดึงค่าได้ถูกไหม

In [ ]:
PLATE_RE = re.compile(r"(?<![ก-๙A-Za-z])((?:[ก-ฮ]{1,3}|[A-Za-z]{1,2})\s*-?\s*\d{3,4})(?!\d)")
TIME_RE = re.compile(r"(\d{1,2})\s*[:.]\s*(\d{2})\s*(?:น\.?)?|(\d{1,2})\s*โมง")

def find_plate(q):
    m = PLATE_RE.search(q)
    if not m:
        return None, q
    key = re.sub(r"[\s-]", "", m.group(1)).upper()
    return key, q.replace(m.group(1), " ")

def find_time(q):
    # คืน (เวลาเริ่ม, เวลาสิ้นสุด, คำอธิบาย) · ข้อมูลเป็น batch → อ้างอิงจาก LATEST
    day0 = LATEST.normalize()
    m = TIME_RE.search(q)
    if "เมื่อวาน" in q:
        a, b, label = day0 - pd.Timedelta(days=1), day0, "เมื่อวาน"
    elif "วันนี้" in q:
        a, b, label = day0, LATEST, "วันนี้"
    else:
        a, b, label = None, LATEST, "ล่าสุด"
    if m:
        hh = int(m.group(1) or m.group(3)); mm = int(m.group(2) or 0)
        base = a if a is not None else day0
        b = base + pd.Timedelta(hours=hh, minutes=mm)
        label = f"{label if a is not None else ''} {hh:02d}:{mm:02d} น.".strip()
        q = q.replace(m.group(0), " ")
    return (a, b, label), q

def grams(s):
    return {s[i:i + 2] for i in range(len(s) - 1)}
GAZ = [(r.name, len(r.name), grams(r.name), r) for r in gaz.itertuples() if len(r.name) >= 2]

def find_place(q, min_score=0.85):
    s = q.lower(); sg = grams(s)
    best = (0.0, None)
    for n, L, g, r in GAZ:
        if n in s:                                           # ตรงเป๊ะ: ชื่อยาวกว่าชนะ
            score = 1.0 + L / 100
        elif L < 4 or len(g & sg) < 0.4 * len(g):           # ชื่อสั้นต้องตรงเป๊ะ (fuzzy จับคำทั่วไปผิด) · กรองเร็วด้วยคู่ตัวอักษร
            continue
        else:
            score = max(difflib.SequenceMatcher(None, n, s[i:i + w]).ratio()   # ช่วงยาวใกล้เคียงชื่อ (พิมพ์ขาด/เกิน 1 ตัว)
                        for w in (L - 1, L, L + 1) for i in range(0, max(1, len(s) - w + 1)))
        if r.level == "prov":
            score += 0.001                                   # เสมอกัน → เลือกจังหวัด
        if score > best[0]:
            best = (score, r)
    if best[0] < min_score:
        return None
    r = best[1]
    return {"level": r.level, "code": r.code, "label": r.label, "matched": r.name, "score": round(min(best[0], 1.0), 2)}

def extract(q):
    plate, rest = find_plate(q)
    when, rest = find_time(rest)
    place = find_place(rest)
    return {"plate": plate, "when": when, "place": place}

for q in ["ตอนนี้รถคันไหนอยู่ภูเกตบ้าง", "EV-0042 อยู่ไหน", "เมื่อวาน EV 0042 ไปไหนมาบ้าง", "รถในเขตจตุจักรตอน 9 โมง"]:
    e = extract(q)
    print(q, "→", e["plate"], "|", e["when"][2], "|", e["place"] and (e["place"]["label"], e["place"]["score"]))

## 4 · สร้างชุดข้อมูลเทรน

**intent** = ประเภทของคำถาม (ถามหาอะไร) — ระบบนี้รู้จัก 8 แบบ:
`cars_in_area` รถในพื้นที่ · `car_status` สถานะรถคันเดียว · `car_track` เส้นทาง · `out_of_zone` รถนอกเขต · `fleet_stats` สถิติภาพรวม · `nearest_charger` จุดชาร์จ · `action_request` ขอให้สั่งการรถ (ปฏิเสธ) · `other` ไม่เกี่ยวข้อง

**ทำไมต้องสังเคราะห์** — ยังไม่มีคำถามจริงจากผู้ใช้ จึงเขียน "แม่แบบ" (template) ไว้หลายแบบต่อ intent แล้วสุ่มเติมชื่อจังหวัด ทะเบียน เวลา คำสุภาพ และจงใจพิมพ์ผิดบ้าง → ได้ ~2,300 ประโยค

**mask** — ก่อนเทรน เปลี่ยนชื่อจริงเป็นคำกลาง (`ภูเก็ต` → `สถานที่` · `EV-0042` → `ทะเบียนรถ`) ให้โมเดลเรียน**รูปประโยค** ไม่ใช่จำชื่อจังหวัด

**จะเพิ่มคำถามแบบใหม่** → เพิ่มประโยคใน dict `T` ใต้ intent ที่ตรง (ใช้ `{place}` `{plate}` `{time}` เป็นช่องเติม) แล้วรันส่วน 4–5 ใหม่

In [ ]:
T = {
 "cars_in_area": [
   "ตอนนี้รถคันไหนอยู่{place}บ้าง", "มีรถกี่คันอยู่ที่{place}", "{place}มีรถอยู่กี่คัน", "รถที่อยู่ใน{place}ตอนนี้",
   "ขอรายชื่อรถใน{place}", "{place}ตอนนี้มีแท็กซี่ไหม", "แท็กซี่ใน{place}มีกี่คัน", "รถว่างใน{place}มีกี่คัน",
   "มีรถวิ่งแถว{place}บ้างไหม", "ใครอยู่{place}บ้าง", "เช็ครถในพื้นที่{place}", "cars in {place_en}",
   "รถคันไหนจอดอยู่{place}", "{place}มีรถมีผู้โดยสารกี่คัน", "ขอดูรถทั้งหมดที่อยู่{place}",
   "จำนวนรถที่อยู่ใน{place}", "{place}มีกี่คัน", "รถเราที่อยู่{place}ตอนนี้มีใครบ้าง", "list รถใน{place}"],
 "car_status": [
   "{plate} อยู่ไหน", "รถ {plate} ตอนนี้อยู่ที่ไหน", "ตำแหน่งล่าสุดของ {plate}", "{plate} วิ่งอยู่หรือจอด",
   "เช็คสถานะรถ {plate}", "ทะเบียน {plate} อยู่จังหวัดอะไร", "{plate} มีผู้โดยสารไหม", "where is {plate}",
   "รถ {plate} ตอนนี้เป็นยังไง", "{plate} ขับไปทางไหน", "หา {plate} ให้หน่อย", "{plate} อยู่แถวไหนแล้ว",
   "{plate} ไปถึงไหนแล้ว", "{plate} ตอนนี้อยู่จังหวัดไหน", "พิกัดรถ {plate}"],
 "car_track": [
   "{plate} ไปไหนมาบ้าง", "เส้นทางของ {plate} {time}", "{plate} วิ่งไปกี่กิโล{time}", "สรุปการเดินทางรถ {plate}",
   "{plate} ผ่านจังหวัดอะไรบ้าง", "ประวัติการวิ่งของ {plate}", "{time} {plate} ไปที่ไหนบ้าง", "{plate} วิ่งทั้งหมดกี่กิโลเมตร",
   "ย้อนดูเส้นทาง {plate}", "{plate} ออกต่างจังหวัดไหม{time}", "track {plate}"],
 "out_of_zone": [
   "{time}มีรถออกนอกเขตกี่คัน", "รถคันไหนออกนอกเขต", "ใครวิ่งออกนอกพื้นที่บ้าง", "รายการรถนอกเขต",
   "มีรถวิ่งออกต่างจังหวัดไหม", "รถที่ออกนอกกรุงเทพและปริมณฑล", "นับรถที่ออกนอกเขต{time}", "รถผิดพื้นที่มีกี่คัน",
   "มีใครขับออกนอกเขตที่อนุญาตไหม", "เช็ครถออกนอกโซน", "รถคันไหนวิ่งไปต่างจังหวัด{time}", "สรุปรถออกนอกพื้นที่{time}",
   "ใครออกนอกเขตบ้าง{time}", "มีรถหลุดโซนไหม"],
 "fleet_stats": [
   "จังหวัดไหนมีรถวิ่งเยอะที่สุด", "สรุปจำนวนรถแยกตามจังหวัด", "ตอนนี้รถออนไลน์กี่คัน", "รถมีผู้โดยสารกี่เปอร์เซ็นต์",
   "มีรถทั้งหมดกี่คัน", "พื้นที่ไหนรถเยอะสุด", "ภาพรวมรถตอนนี้", "จัดอันดับจังหวัดตามจำนวนรถ", "สถิติรถ{time}",
   "รถวิ่งอยู่กี่คันจอดกี่คัน", "ทั้งประเทศมีรถออนไลน์กี่คัน", "สรุปภาพรวมทั้งหมด{time}", "ตอนนี้มีรถวิ่งกี่คัน",
   "อัตรารถมีผู้โดยสารตอนนี้", "จังหวัดที่มีรถมากที่สุด 5 อันดับ"],
 "nearest_charger": [
   "{plate} ใกล้จุดชาร์จไหน", "สถานีชาร์จใกล้{place}", "จุดชาร์จ EV ที่ใกล้รถ {plate} ที่สุด", "หาที่ชาร์จแถว{place}",
   "แถว{place}มีปั๊มชาร์จไหม", "รถ {plate} ควรไปชาร์จที่ไหน", "ที่ชาร์จไฟใกล้ๆ{place}", "charger near {place_en}"],
 "action_request": [
   "ตัดสัญญาณรถ {plate} ให้หน่อย", "ล็อครถ {plate}", "สั่งดับเครื่อง {plate}", "ปิดรถ {plate} เลย",
   "ส่งข้อความหาคนขับ {plate}", "บล็อกรถคันนี้ {plate}", "หยุดรถ {plate}", "ล็อคล้อ {plate}", "ยึดรถ {plate}",
   "ตัดไฟรถ {plate}", "ปลดล็อครถ {plate}"],
 "other": [
   "สวัสดี", "อากาศวันนี้เป็นยังไง", "ช่วยเขียนอีเมลให้หน่อย", "ราคาหุ้นวันนี้", "คุณเป็นใคร", "ขอบคุณ",
   "เล่าเรื่องตลกหน่อย", "แปลภาษาอังกฤษให้หน่อย", "วันนี้วันอะไร", "กินอะไรดี", "ราคาน้ำมันวันนี้",
   "บอกข่าวล่าสุด", "ช่วยคิดเลขให้หน่อย", "เพลงเพราะๆ"],
}
TIMES = ["", "วันนี้", "เมื่อวาน", "ตอนนี้", "ตอน 9 โมง", "เมื่อ 14:30"]
plates = pos.plate.drop_duplicates().sample(300, random_state=1).tolist()
places = gaz[gaz.level == "prov"].sample(frac=1, random_state=1)
pen = dict(zip(prov.ppc.str[2:], prov.pen))

PRE = ["", "", "", "ขอ", "ช่วยบอกหน่อย ", "อยากรู้ว่า ", "รบกวนเช็ค ", "แอดมิน "]
POST = ["", "", "", " ครับ", " คะ", " หน่อย", " ด่วน", " ?", " นะ"]

def typo(s, p=0.15):
    # ทำให้พิมพ์ผิดเล็กน้อย: ลบ/สลับ/ซ้ำ 1 ตัวอักษร
    if random.random() > p or len(s) < 4:
        return s
    i = random.randrange(1, len(s) - 1); k = random.choice("dsr")
    return s[:i] + s[i + 1:] if k == "d" else s[:i] + s[i + 1] + s[i] + s[i + 2:] if k == "s" else s[:i] + s[i] + s[i:]

random.seed(7)
rows = []
for intent, tpls in T.items():
    for ti, t in enumerate(tpls):
        for _ in range(25):
            pl = places.sample(1, random_state=random.randrange(10**6)).iloc[0]
            q = t.format(place=typo(pl["name"]), place_en=pen.get(pl.code, pl["name"]),
                         plate=random.choice(plates), time=random.choice(TIMES))
            q = random.choice(PRE) + q + random.choice(POST)    # คำสุภาพ/คำเสริม ให้ประโยคหลากหลาย
            rows.append((re.sub(r"\s+", " ", q).strip(), intent, f"{intent}#{ti}"))
train_df = pd.DataFrame(rows, columns=["q", "intent", "tpl"]).drop_duplicates("q")

def mask(q):
    # แทนค่าที่ดึงได้ด้วยคำกลาง ก่อนส่งเข้า classifier
    e = extract(q)
    _, rest = find_plate(q)
    if e["plate"]:
        rest = rest + " ทะเบียนรถ"
    _, rest = find_time(rest)
    if e["place"]:
        rest = rest.lower().replace(e["place"]["matched"], " สถานที่ ")
        if "สถานที่" not in rest:
            rest += " สถานที่"
    return re.sub(r"\s+", " ", rest).strip()

train_df["masked"] = train_df.q.map(mask)
print(len(train_df), "ประโยค ·", train_df.intent.value_counts().to_dict())
train_df.sample(6, random_state=3)[["q", "masked", "intent"]]

## 5 · เทรนตัวจัดประเภทคำถาม

**ขั้นที่ 1 — แปลงประโยคเป็นตัวเลข (embedding)**
คอมพิวเตอร์เข้าใจตัวเลข ไม่เข้าใจภาษา · **backbone** (`paraphrase-multilingual-MiniLM`) เป็นโมเดลที่ฝึกมาแล้วกับข้อความหลายภาษา แปลงประโยคเป็นเวกเตอร์ 384 ตัวเลข — ประโยคความหมายคล้ายกันจะได้ตัวเลขใกล้กัน
เป็นตัวเดียวกับที่ใช้จัดหมวดสินค้า ITEC · ครั้งแรกบน Colab จะดาวน์โหลด ~470 MB (1–2 นาที)

**ขั้นที่ 2 — เทรน classifier (Logistic Regression)** บนเวกเตอร์เหล่านั้น ให้รู้ว่าเวกเตอร์แบบไหนเป็น intent อะไร — ใช้เวลาไม่กี่วินาที ไม่ต้องใช้ GPU

**อ่านผลยังไง (ตาราง classification report)**
- **precision** — ที่โมเดลทายว่าเป็น X ถูกกี่ %
- **recall** — ข้อที่เป็น X จริง โมเดลหาเจอกี่ %
- **f1** — ค่าเฉลี่ยของสองตัวบน · **accuracy** — ถูกกี่ % ของทั้งหมด

ชุดทดสอบใช้ **template ที่โมเดลไม่เคยเห็น** (เหมือนผู้ใช้พิมพ์รูปประโยคใหม่) — คะแนนจึงต่ำกว่าการสุ่มแบ่งธรรมดาแต่ใกล้ความจริงกว่า · intent ที่ recall ต่ำ = ต้องเพิ่ม template แบบนั้น

cell ถัดไปทดสอบกับ **คำถามที่เขียนเอง 20 ข้อ** (`HAND`) แล้วแสดงข้อที่ทายผิด

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

BACKBONE = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # ตัวเดียวกับ scripts/itec/category (cache ไว้แล้ว)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
enc = SentenceTransformer(BACKBONE, device=DEVICE)          # Colab ดาวน์โหลดโมเดลเองครั้งแรก (~470 MB)
print("device:", DEVICE)

t0 = time.time()
X = enc.encode(train_df.masked.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
print(f"encode {len(X):,} ประโยค {time.time() - t0:.0f} วินาที")

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)       # แบ่งตาม template
tr, te = next(gss.split(X, train_df.intent, groups=train_df.tpl))
clf = LogisticRegression(C=8, max_iter=2000, class_weight="balanced").fit(X[tr], train_df.intent.iloc[tr])
print(classification_report(train_df.intent.iloc[te], clf.predict(X[te]), digits=3, zero_division=0))

In [ ]:
# เทรนใหม่ด้วยข้อมูลทั้งหมดสำหรับใช้งาน + ชุดทดสอบที่เขียนเอง (ไม่ได้มาจาก template)
clf = LogisticRegression(C=8, max_iter=2000, class_weight="balanced").fit(X, train_df.intent)

HAND = [
 ("ภูเก็ตตอนนี้มีรถเรามั้ย", "cars_in_area"), ("แถวบางนามีแท็กซี่ว่างกี่คันครับ", "cars_in_area"),
 ("ขอลิสต์รถที่อยู่เชียงใหม่", "cars_in_area"), ("นนทบุรีมีรถกี่คันตอนนี้", "cars_in_area"),
 ("ทอ-4141 ตอนนี้ไปถึงไหนแล้ว", "car_status"), ("รถทะเบียน ทอ 8675 จอดอยู่หรือเปล่า", "car_status"),
 ("ช่วยหาตำแหน่ง E-1234", "car_status"),
 ("เมื่อวานทอ-4141 วิ่งไปไหนบ้าง", "car_track"), ("ทอ-8571 วันนี้วิ่งไปกี่โล", "car_track"),
 ("วันนี้มีใครออกนอกเขตบ้าง", "out_of_zone"), ("รถที่วิ่งออกต่างจังหวัดวันนี้", "out_of_zone"),
 ("ตอนนี้รถวิ่งอยู่ทั้งหมดกี่คัน", "fleet_stats"), ("จังหวัดไหนรถเยอะที่สุดตอนนี้", "fleet_stats"),
 ("ทอ-4141 แบตใกล้หมด ชาร์จที่ไหนใกล้สุด", "nearest_charger"), ("ที่ชาร์จรถไฟฟ้าแถวหาดใหญ่", "nearest_charger"),
 ("ช่วยตัดไฟรถ ทอ-4141 ที", "action_request"), ("ล็อคล้อรถคันนี้ ทอ-8675", "action_request"),
 ("หวัดดีครับ", "other"), ("ฝนจะตกไหมวันนี้", "other"), ("ขอสูตรต้มยำ", "other"),
]
hand = pd.DataFrame(HAND, columns=["q", "intent"])
Xh = enc.encode(hand.q.map(mask).tolist(), normalize_embeddings=True)
proba = clf.predict_proba(Xh)
hand["pred"] = clf.classes_[proba.argmax(1)]
hand["conf"] = proba.max(1).round(2)
print(f"ถูก {(hand.pred == hand.intent).mean():.0%} ของ {len(hand)} ข้อ")
hand[hand.pred != hand.intent]

## 6 · ตัวทำงาน (executor) — ค้นข้อมูลจริง

เมื่อรู้ intent และ slot แล้ว ส่วนนี้คือ "คนไปค้นตาราง" — เป็นโค้ด pandas ธรรมดา **ตัวเลขทุกตัวในคำตอบมาจากที่นี่** (ไม่มีโมเดลไหนเดาตัวเลข)

| ฟังก์ชัน | ทำอะไร |
|---|---|
| `latest_at(t)` | ตำแหน่งล่าสุดของแต่ละคัน ณ เวลา t (นับเฉพาะจุดใน 15 นาทีก่อนหน้า — เก่ากว่านั้นถือว่าออฟไลน์) |
| `x_cars_in_area` | กรองจาก `latest_at` เฉพาะจังหวัด/อำเภอที่ถาม |
| `x_car_status` | จุดล่าสุดของรถคันนั้น |
| `x_car_track` | รวมระยะทาง (สูตร haversine = ระยะบนผิวโลก) ไม่นับช่วงที่สัญญาณหายเกิน 15 นาที |
| `x_out_of_zone` | รถที่มีจุดนอกกรุงเทพฯ และปริมณฑล |
| `x_fleet_stats` | นับรถออนไลน์ แยกสถานะ และ 5 จังหวัดที่รถเยอะสุด |
| `x_nearest_charger` | สถานีชาร์จที่ใกล้รถ (หรือใกล้กลางจังหวัด) ที่สุด 3 แห่ง |

In [ ]:
STALE = pd.Timedelta(minutes=15)
ZONE = {"10", "11", "12", "13", "73", "74"}             # กรุงเทพฯ และปริมณฑล
chargers = json.load(open(OUT / "ev_chargers_osm.json", encoding="utf-8"))
chg = pd.DataFrame([{"lat": (e.get("lat") or e["center"]["lat"]), "lon": (e.get("lon") or e["center"]["lon"]),
                     "name": e.get("tags", {}).get("name", "") or "(ไม่มีชื่อ)",
                     "op": e.get("tags", {}).get("operator") or e.get("tags", {}).get("brand") or ""}
                    for e in chargers if "lat" in e or "center" in e])

def hav_km(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(a))

def latest_at(t):
    w = pos[(pos.gps_time <= t) & (pos.gps_time > t - STALE)]
    return w.groupby("car_id").tail(1)

def car_rows(plate_key):
    return pos[pos.plate_key.str.upper() == plate_key]

def x_cars_in_area(place, when):
    if not place:
        return {"error": "ไม่พบชื่อสถานที่ในคำถาม"}
    now = latest_at(when[1])
    col = "prov_code" if place["level"] == "prov" else "dist_code"
    hit = now[now[col] == place["code"]]
    return {"as_of": when[1], "place": place["label"], "n": len(hit), "by_status": hit.status.value_counts().to_dict(),
            "cars": hit[["car_id", "plate", "status", "dist_th"]].head(20).to_dict("records")}

def x_car_status(plate, when):
    r = car_rows(plate) if plate else pd.DataFrame()
    if r.empty:
        return {"error": f"ไม่พบรถทะเบียน {plate or '(ไม่ได้ระบุ)'}"}
    r = r[r.gps_time <= when[1]]
    if r.empty:
        return {"error": "ไม่มีข้อมูลก่อนเวลาที่ถาม"}
    x = r.iloc[-1]
    return {"as_of": when[1], "plate": x.plate, "car_id": x.car_id, "time": x.gps_time, "age_min": round((when[1] - x.gps_time).total_seconds() / 60),
            "status": x.status, "speed": x.speed, "heading": x.direction, "where": f"{x.dist_th}, {x.prov_th}", "lat": x.lat, "lon": x.lon}

def x_car_track(plate, when):
    r = car_rows(plate) if plate else pd.DataFrame()
    if r.empty:
        return {"error": f"ไม่พบรถทะเบียน {plate or '(ไม่ได้ระบุ)'}"}
    a = when[0] if when[0] is not None else LATEST.normalize()
    r = r[(r.gps_time >= a) & (r.gps_time <= when[1]) & (r.signal != "V")]
    if len(r) < 2:
        return {"error": "ข้อมูลในช่วงนี้ไม่พอ"}
    gap = r.gps_time.diff() <= STALE                       # ไม่นับระยะข้ามช่วงที่สัญญาณหาย
    km = hav_km(r.lat.shift(), r.lon.shift(), r.lat, r.lon)[gap].sum()
    provs = r.groupby("prov_th", sort=False).size()
    return {"plate": r.plate.iloc[0], "from": r.gps_time.iloc[0], "to": r.gps_time.iloc[-1], "km": round(float(km), 1),
            "max_speed": r.speed.max(), "provinces": list(provs.index), "start": f"{r.dist_th.iloc[0]}, {r.prov_th.iloc[0]}",
            "end": f"{r.dist_th.iloc[-1]}, {r.prov_th.iloc[-1]}"}

def x_out_of_zone(when):
    a = when[0] if when[0] is not None else LATEST.normalize()
    r = pos[(pos.gps_time >= a) & (pos.gps_time <= when[1]) & (pos.signal != "V") & ~pos.prov_code.isin(ZONE)]
    g = r.groupby(["car_id", "plate"]).agg(provinces=("prov_th", lambda s: ", ".join(s.unique()[:3])), points=("prov_th", "size"))
    g = g.sort_values("points", ascending=False).reset_index()
    return {"from": a, "to": when[1], "n": len(g), "cars": g.head(20).to_dict("records")}

def x_fleet_stats(when):
    now = latest_at(when[1])
    top = now.groupby("prov_th").size().sort_values(ascending=False).head(5)
    return {"as_of": when[1], "online": len(now), "by_status": now.status.value_counts().to_dict(), "top_prov": top.to_dict()}

def x_nearest_charger(plate, place, when, k=3):
    if plate:
        s = x_car_status(plate, when)
        if "error" in s:
            return s
        lat, lon, ref = s["lat"], s["lon"], f"รถ {s['plate']} ({s['where']})"
    elif place:
        code = place["code"][:2]
        lat, lon = cent.loc[code, "centroid_lat"], cent.loc[code, "centroid_lon"]
        ref = f"กลางจังหวัด {name_of[code]}"
    else:
        return {"error": "ต้องระบุทะเบียนรถหรือสถานที่"}
    d = chg.assign(km=hav_km(lat, lon, chg.lat, chg.lon)).nsmallest(k, "km")
    return {"ref": ref, "items": d[["name", "op", "km"]].round(1).to_dict("records"),
            "note": "ข้อมูลสถานีชาร์จจาก OpenStreetMap มีแค่บางส่วน (306 จุด)"}

## 7 · รวมเป็น `ask()` — ถามได้เลย

ลำดับใน `ask(คำถาม)`: ดึง slot (ส่วน 3) → ทาย intent (ส่วน 5) → ถ้ามั่นใจต่ำกว่า `MIN_CONF` (0.45) **ถามกลับ ไม่เดา** → เรียก executor (ส่วน 6) → สร้างประโยคตอบจากแม่แบบ

**ลองถามเอง:** เพิ่ม cell ใหม่ (ปุ่ม `+ Code`) แล้วพิมพ์
```python
print(ask("ตอนนี้รถคันไหนอยู่ชลบุรีบ้าง", verbose=True))
```
`verbose=True` จะแสดงว่าระบบเข้าใจคำถามยังไง (intent · ความมั่นใจ · slot) — ใช้ดูว่าผิดตรงไหนเวลาตอบไม่ถูก

In [ ]:
MIN_CONF = 0.45            # มั่นใจต่ำกว่านี้ = ถามกลับ ไม่เดา

def fmt_t(t):
    return pd.Timestamp(t).strftime("%d/%m %H:%M")

def answer(intent, r):
    if "error" in r:
        return f"ขออภัย {r['error']}"
    if intent == "cars_in_area":
        if not r["n"]:
            return f"ไม่มีรถออนไลน์ใน{r['place']} (ข้อมูล ณ {fmt_t(r['as_of'])})"
        st = " · ".join(f"{k} {v}" for k, v in r["by_status"].items())
        lst = ", ".join(c["plate"] for c in r["cars"][:10])
        more = f" และอีก {r['n'] - 10} คัน" if r["n"] > 10 else ""
        return f"{r['place']} มีรถออนไลน์ {r['n']} คัน ({st}) — {lst}{more} · ข้อมูล ณ {fmt_t(r['as_of'])}"
    if intent == "car_status":
        return (f"{r['plate']} อยู่ที่ {r['where']} · {r['status']} {r['speed']:.0f} กม./ชม. ทิศ {r['heading']:.0f}° "
                f"· จุดล่าสุด {fmt_t(r['time'])} ({r['age_min']} นาทีก่อนเวลาที่ถาม)")
    if intent == "car_track":
        return (f"{r['plate']} ช่วง {fmt_t(r['from'])}–{fmt_t(r['to'])} วิ่งประมาณ {r['km']} กม. (เร็วสุด {r['max_speed']:.0f} กม./ชม.) "
                f"จาก {r['start']} ถึง {r['end']} · ผ่าน {', '.join(r['provinces'])}")
    if intent == "out_of_zone":
        lst = ", ".join(f"{c['plate']} ({c['provinces']})" for c in r["cars"][:8])
        return f"ช่วง {fmt_t(r['from'])}–{fmt_t(r['to'])} มีรถออกนอกกรุงเทพฯ และปริมณฑล {r['n']} คัน" + (f" — {lst}" if lst else "")
    if intent == "fleet_stats":
        st = " · ".join(f"{k} {v}" for k, v in r["by_status"].items())
        top = ", ".join(f"{k} {v}" for k, v in r["top_prov"].items())
        return f"ณ {fmt_t(r['as_of'])} รถออนไลน์ {r['online']:,} คัน ({st}) · จังหวัดที่รถเยอะสุด: {top}"
    if intent == "nearest_charger":
        lst = " · ".join(f"{i['name']}{' (' + i['op'] + ')' if i['op'] else ''} {i['km']} กม." for i in r["items"])
        return f"สถานีชาร์จใกล้{r['ref']}: {lst} — {r['note']}"
    return ""

def ask(q, verbose=False):
    e = extract(q)
    p = clf.predict_proba(enc.encode([mask(q)], normalize_embeddings=True))[0]
    intent, conf = clf.classes_[p.argmax()], float(p.max())
    if verbose:
        print(f"  intent={intent} ({conf:.2f}) plate={e['plate']} place={e['place'] and e['place']['label']} when={e['when'][2]}")
    if conf < MIN_CONF:
        return "ไม่แน่ใจว่าถามเรื่องอะไร ลองถามเช่น 'ตอนนี้รถคันไหนอยู่ภูเก็ต' หรือ 'ทอ-4141 อยู่ไหน'"
    if intent == "action_request":
        return "ระบบนี้ตอบคำถามได้อย่างเดียว ไม่สามารถสั่งการรถ (ตัดสัญญาณ/ล็อก/ดับเครื่อง) ได้"
    if intent == "other":
        return "ตอบได้เฉพาะคำถามเรื่องรถแท็กซี่ EV7 เช่น ตำแหน่งรถ รถในพื้นที่ เส้นทาง รถนอกเขต สถานีชาร์จ"
    run = {"cars_in_area": lambda: x_cars_in_area(e["place"], e["when"]),
           "car_status": lambda: x_car_status(e["plate"], e["when"]),
           "car_track": lambda: x_car_track(e["plate"], e["when"]),
           "out_of_zone": lambda: x_out_of_zone(e["when"]),
           "fleet_stats": lambda: x_fleet_stats(e["when"]),
           "nearest_charger": lambda: x_nearest_charger(e["plate"], e["place"], e["when"])}[intent]
    return answer(intent, run())

demo_plate = pos.plate.value_counts().index[0]      # ทะเบียนที่มีข้อมูลมากสุด ใช้ทดลอง
for q in ["ตอนนี้รถคันไหนอยู่ภูเกตบ้าง", f"{demo_plate} อยู่ไหน", f"วันนี้ {demo_plate} ไปไหนมาบ้าง",
          "วันนี้มีรถออกนอกเขตกี่คัน", "จังหวัดไหนมีรถเยอะที่สุด", "ที่ชาร์จแถวเชียงใหม่",
          f"ตัดสัญญาณรถ {demo_plate} ให้หน่อย", "ขอสูตรต้มยำ"]:
    print("Q:", q); print("A:", ask(q, verbose=True)); print()

In [ ]:
# บันทึกโมเดล (backbone ไม่ต้องบันทึก — โหลดจากชื่อ)
import joblib
SAVE = (DRIVE if IN_COLAB else DATA) / "qa_intent.joblib"     # Colab: เก็บลง Drive (เครื่อง Colab ถูกล้างเมื่อปิด)
joblib.dump({"clf": clf, "backbone": BACKBONE, "min_conf": MIN_CONF, "trained": str(pd.Timestamp.now())}, SAVE)
print("บันทึก", SAVE)

## 8 · (ทางเลือก) ใช้ LLM รันเอง + สอนเพิ่ม (fine-tune)

### ทำไมต้องมีส่วนนี้
ส่วน 1–7 ตอบได้เฉพาะ 8 แบบที่สอนไว้ · ถ้าผู้ใช้ถามยาก ๆ เช่น *"รถที่อยู่ชลบุรีแต่เมื่อวานอยู่กรุงเทพมีกี่คัน"* ตัวจัดประเภทจะทำไม่ได้ · **LLM** (Large Language Model — โมเดลภาษาขนาดใหญ่แบบ ChatGPT) เข้าใจประโยคซับซ้อนได้ดีกว่า

### คำศัพท์ที่ต้องรู้ก่อน
| คำ | ความหมายแบบง่าย |
|---|---|
| **LLM โอเพนซอร์ส** | LLM ที่ดาวน์โหลดตัวโมเดลมารันเองได้ (ไม่ต้องส่งข้อมูลไปบริษัทอื่น) เช่น Qwen · Typhoon |
| **fine-tune** | สอนโมเดลที่ฉลาดอยู่แล้วให้ทำงานเฉพาะของเรา ด้วยตัวอย่างไม่กี่พันข้อ |
| **LoRA** | วิธี fine-tune แบบประหยัด: ไม่แก้โมเดลทั้งตัว (พันล้านตัวเลข) แต่เพิ่ม "ชิ้นส่วนเสริม" เล็ก ๆ (~1% ของโมเดล) แล้วเทรนแค่ชิ้นนั้น → ใช้หน่วยความจำน้อย ไฟล์ผลลัพธ์เล็ก (หลักสิบ MB) |
| **4-bit (quantization)** | บีบโมเดลให้เล็กลง ~4 เท่า (เก็บตัวเลขด้วย 4 บิตแทน 16) → ใส่ GPU ฟรีของ Colab ได้ · ความแม่นลดลงเล็กน้อย |
| **GPU / T4** | ชิปที่คำนวณงาน AI เร็วกว่า CPU หลายสิบเท่า · Colab ฟรีให้ T4 (หน่วยความจำ 15 GB) ใช้ได้จำกัดชั่วโมง |
| **token** | ชิ้นส่วนของข้อความที่โมเดลอ่าน (ประมาณคำ/พยางค์) |
| **loss** | ตัวเลขบอกว่าโมเดลยังทายผิดมากแค่ไหน — ระหว่างเทรนควร**ลดลง** |

### เราสอน LLM ให้ทำอะไร
**ไม่ได้สอนให้จำตำแหน่งรถ** (ข้อมูลเปลี่ยนทุกนาที สอนไปก็ล้าสมัย) · สอนให้ **แปลงคำถาม → คำสั่ง JSON** แล้วให้ executor ส่วน 6 ไปค้นข้อมูลจริงเหมือนเดิม

```
คำถาม: ภูเกตตอนนี้มีรถกี่คัน
JSON:  {"intent": "cars_in_area", "plate": null, "place": "ภูเก็ต", "time": "ล่าสุด"}
```
→ LLM ทำหน้าที่แทนส่วน 3 + 5 (เข้าใจคำถาม) · ส่วน 6 + 7 (ค้นข้อมูล + ตอบ) ใช้ของเดิม

### ขั้นตอนทั้งหมด (แต่ละขั้นคือ 1 cell ข้างล่าง)
| ขั้น | ทำอะไร | เวลาโดยประมาณ (T4) |
|---|---|---|
| 8.1 | เปลี่ยนเครื่องเป็น GPU + ตรวจ | 1 นาที |
| 8.2 | เตรียมตัวอย่างสอน (คำถาม → JSON) | ไม่กี่วินาที |
| 8.3 | ติดตั้งไลบรารี | 1–2 นาที |
| 8.4 | ดาวน์โหลด + โหลดโมเดลแบบ 4-bit | 2–5 นาที |
| 8.5 | ติด LoRA | ไม่กี่วินาที |
| 8.6 | แปลงตัวอย่างเป็น token | ไม่กี่วินาที |
| 8.7 | เทรน | 10–20 นาที |
| 8.8 | ทดสอบ เทียบกับตัวจัดประเภทส่วน 5 | 1–2 นาที |
| 8.9 | บันทึกลง Drive + วิธีโหลดกลับมาใช้ | ไม่กี่วินาที |
| 8.10 | ถามด้วย LLM ผ่าน `ask_llm()` | — |

เวลาเป็นค่าประมาณ `[อนุมาน — ยังไม่ได้ทดสอบบน GPU จริง]`

### 8.1 · เปลี่ยนเครื่องเป็น GPU และเปิดส่วนนี้

1. เมนูบน **Runtime → Change runtime type**
2. **Hardware accelerator** เลือก **T4 GPU** → **Save**
3. ⚠️ การเปลี่ยน runtime = **เริ่มเครื่องใหม่ ตัวแปรทั้งหมดหาย** → ต้องรันส่วน 0–7 ใหม่ทั้งหมด (เมนู **Runtime → Run all** ได้ หรือรันทีละ cell)
4. แก้ `RUN_LLM = False` เป็น `RUN_LLM = True` ใน cell ข้างล่าง แล้วรัน
5. ผลที่ถูก: `GPU: Tesla T4 ...` · ถ้าขึ้น `ไม่มี GPU` แปลว่าข้อ 1–2 ยังไม่สำเร็จ หรือโควตา GPU ฟรีวันนี้หมด (ลองใหม่ภายหลัง)

**เลือกโมเดล (`LLM_ID`)** — ค่าเริ่มต้น `Qwen/Qwen2.5-1.5B-Instruct` (1.5 พันล้านพารามิเตอร์ ขนาดที่ T4 ฟรีเทรนไหว รองรับหลายภาษารวมไทย)
ถ้าจะใช้รุ่นที่ฝึกภาษาไทยมาโดยเฉพาะ (เช่น Typhoon) ค้นชื่อรุ่นใน huggingface.co แล้วใส่แทน — **ตรวจ License ในหน้าโมเดลก่อน** ว่าใช้เชิงพาณิชย์ได้ไหม · รุ่น 7B ขึ้นไปอาจไม่พอบน T4

In [ ]:
RUN_LLM = False          # ← เปลี่ยนเป็น True หลังทำขั้น 1–3 ข้างบน
LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"

if RUN_LLM:
    assert torch.cuda.is_available(), "ไม่มี GPU — Runtime → Change runtime type → T4 GPU แล้วรันส่วน 0–7 ใหม่"
    print("GPU:", torch.cuda.get_device_name(0), f"· หน่วยความจำ {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
else:
    print("ข้ามส่วน 8 (RUN_LLM = False) · GPU:", torch.cuda.is_available())

### 8.2 · เตรียมตัวอย่างสอน (คำถาม → JSON)

ใช้ประโยคจากส่วน 4 (`train_df`) มาสร้างคู่ **โจทย์ (prompt)** กับ **เฉลย (completion)** 2,000 คู่
- prompt = `แปลงคำถามเป็นคำสั่ง JSON / คำถาม: ... / JSON:`
- completion = JSON ที่ถูกต้อง — สร้างจาก intent ที่รู้อยู่แล้ว + slot ที่ส่วน 3 ดึงได้

cell นี้รันได้แม้ไม่มี GPU — ดูตัวอย่าง 3 คู่ว่าหน้าตาถูกต้องก่อนเทรน (**ถ้าเฉลยผิด โมเดลก็จะเรียนผิด**)

In [ ]:
def to_target(q, intent):
    e = extract(q)
    return json.dumps({"intent": intent, "plate": e["plate"], "place": e["place"] and e["place"]["label"],
                       "time": e["when"][2]}, ensure_ascii=False)

PROMPT = "แปลงคำถามเป็นคำสั่ง JSON\nคำถาม: {q}\nJSON:"
sft = [{"prompt": PROMPT.format(q=r.q), "completion": " " + to_target(r.q, r.intent)}
       for r in train_df.sample(min(2000, len(train_df)), random_state=0).itertuples()]
print(len(sft), "คู่")
for x in sft[:3]:
    print("-" * 60); print(x["prompt"] + x["completion"])

### 8.3 · ติดตั้งไลบรารี

Colab มี `transformers` และ `torch` อยู่แล้ว · ต้องติดตั้งเพิ่ม 3 ตัว:
- `peft` — ทำ LoRA
- `bitsandbytes` — บีบโมเดลเป็น 4-bit
- `accelerate` — จัดการวางโมเดลลง GPU

ถ้าขึ้นข้อความแนะนำให้ **Restart session** หลังติดตั้ง → กด Restart แล้วรันส่วน 0–7 + 8.1–8.2 ใหม่

In [ ]:
if RUN_LLM:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "accelerate"], check=True)
    import peft, bitsandbytes
    print("peft", peft.__version__, "· bitsandbytes", bitsandbytes.__version__)
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.4 · ดาวน์โหลดและโหลดโมเดลแบบ 4-bit

- ครั้งแรกจะดาวน์โหลดโมเดลจาก Hugging Face (~3 GB สำหรับ 1.5B) — รอได้
- `BitsAndBytesConfig(load_in_4bit=True)` = บีบให้เล็กลงตอนโหลด
- `float16` เพราะ T4 ไม่รองรับ `bfloat16`
- **tokenizer** = ตัวตัดข้อความเป็น token ของโมเดลรุ่นนี้ (ต้องใช้คู่กับโมเดลเดียวกันเสมอ)

ผลที่ถูก: แสดงหน่วยความจำ GPU ที่ใช้ (~1.5–2.5 GB)

In [ ]:
if RUN_LLM:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(LLM_ID)
    tok.pad_token = tok.pad_token or tok.eos_token          # ตัวเติมช่องว่างให้ประโยคยาวเท่ากันในชุดเดียว
    model = AutoModelForCausalLM.from_pretrained(
        LLM_ID, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=torch.float16))
    print(f"โหลดแล้ว · ใช้หน่วยความจำ GPU {torch.cuda.memory_allocated() / 1e9:.1f} GB")
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.5 · ติด LoRA

- `r=16` = ขนาดของชิ้นส่วนเสริม (ใหญ่ขึ้น = เรียนได้มากขึ้นแต่ช้าลง · 8–32 เป็นค่าทั่วไป)
- `target_modules` = ส่วนของโมเดลที่ติดชิ้นเสริม (ส่วน attention ของ Qwen/Llama ใช้ชื่อ `q_proj k_proj v_proj o_proj` — โมเดลตระกูลอื่นอาจใช้ชื่ออื่น ถ้า error เรื่องชื่อ ให้ดูชื่อจาก `print(model)`)

ผลที่ถูก: `trainable params: ... || all params: ... || trainable%: ~0.5–1` = เทรนแค่ ~1% ของโมเดล

In [ ]:
if RUN_LLM:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model)           # เตรียมโมเดล 4-bit ให้เทรนได้
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    model.print_trainable_parameters()
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.6 · แปลงตัวอย่างเป็น token

โมเดลเรียนจากตัวเลข จึงต้องแปลงข้อความเป็น token id
**จุดสำคัญ:** ใส่ label `-100` ให้ส่วนโจทย์ = บอกโมเดลว่า "ไม่ต้องเรียนการพิมพ์โจทย์ ให้เรียนแค่การตอบ JSON"
ท้ายเฉลยเติม `eos_token` (สัญลักษณ์จบข้อความ) ให้โมเดลรู้ว่าตอบจบแล้ว ไม่พิมพ์ต่อเรื่อย ๆ

In [ ]:
if RUN_LLM:
    def encode_ex(ex, max_len=256):
        p = tok(ex["prompt"], add_special_tokens=False)["input_ids"]
        c = tok(ex["completion"] + tok.eos_token, add_special_tokens=False)["input_ids"]
        ids = (p + c)[:max_len]
        return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": ([-100] * len(p) + c)[:max_len]}

    train_ds = [encode_ex(x) for x in sft]
    x = train_ds[0]
    print("token ทั้งหมด", len(x["input_ids"]), "· ส่วนที่เรียน (ไม่ใช่ -100)", sum(l != -100 for l in x["labels"]))
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.7 · เทรน

| ค่า | ความหมาย |
|---|---|
| `per_device_train_batch_size=8` | ป้อนทีละ 8 ตัวอย่าง (ถ้า error `CUDA out of memory` ลดเป็น 4) |
| `gradient_accumulation_steps=2` | สะสม 2 รอบก่อนปรับโมเดล = เหมือน batch 16 |
| `num_train_epochs=1` | ดูตัวอย่างทั้งหมด 1 รอบ (2,000 ข้อ ≈ 125 รอบการปรับ) |
| `learning_rate=2e-4` | ขนาดก้าวในการปรับ (ค่ามาตรฐานของ LoRA) |
| `fp16=True` | คำนวณด้วยความละเอียดครึ่งหนึ่ง เร็วขึ้นบน T4 |

**ระหว่างเทรน** ตาราง `Step / Training Loss` จะขึ้นทุก 20 รอบ — **loss ควรลดลง** (เช่น จาก ~2 ลงไปต่ำกว่า 0.2) · ถ้าไม่ลดเลย = ตั้งค่าผิดหรือข้อมูลผิด
⚠️ อย่าปิดแท็บ / ปล่อยว่างนาน — Colab ฟรีตัดการเชื่อมต่อเมื่อไม่มีการใช้งาน (ทุกอย่างหาย ต้องเริ่มใหม่)

In [ ]:
if RUN_LLM:
    from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
    trainer = Trainer(
        model=model, train_dataset=train_ds,
        data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
        args=TrainingArguments(output_dir="/content/lora_taxi", per_device_train_batch_size=8,
                               gradient_accumulation_steps=2, num_train_epochs=1, learning_rate=2e-4,
                               fp16=True, logging_steps=20, save_strategy="no", report_to=[]))
    out = trainer.train()
    m = out.metrics
    print(f"เทรนเสร็จ {m['train_runtime'] / 60:.1f} นาที · {out.global_step} รอบการปรับ · loss เฉลี่ย {m['train_loss']:.3f}")
    if m["train_runtime"] < 60:
        print("⚠️ เร็วผิดปกติ — ตรวจว่า train_ds มีข้อมูล (len(train_ds)) และรันขั้น 8.2–8.6 ครบ")
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.8 · ทดสอบ — เทียบกับตัวจัดประเภทส่วน 5

`llm_parse(คำถาม)` ให้โมเดลเขียน JSON (`do_sample=False` = ตอบแบบเดิมทุกครั้ง ไม่สุ่ม) แล้วแปลงเป็น dict · ถ้าโมเดลเขียน JSON ผิดรูปแบบ จะคืน `intent: None` (ระบบจริงต้องถามกลับ)
ทดสอบกับคำถามที่เขียนเอง 20 ข้อ (`hand`) — คอลัมน์ `pred` = ตัวจัดประเภทส่วน 5 · `llm` = LLM

**ตัดสินใจยังไง:** ถ้า LLM ไม่ได้ถูกมากกว่าชัดเจน → ใช้ตัวจัดประเภทส่วน 5 ต่อ (เร็วกว่า ไม่ต้องใช้ GPU) · LLM คุ้มเมื่อมีคำถามซับซ้อนที่ส่วน 5 ทำไม่ได้ — ควรเพิ่มคำถามแบบนั้นลง `HAND` เพื่อวัดด้วย

In [ ]:
if RUN_LLM:
    model.eval()

    def llm_parse(q):
        x = tok(PROMPT.format(q=q), return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**x, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            return json.loads(txt.splitlines()[0])
        except Exception:
            return {"intent": None, "raw": txt}

    got = [llm_parse(q) for q in hand.q]
    hand["llm"] = [g.get("intent") for g in got]
    print(f"LLM ถูก {(hand.llm == hand.intent).mean():.0%} · ตัวจัดประเภทส่วน 5 ถูก {(hand.pred == hand.intent).mean():.0%}")
    display(hand[["q", "intent", "pred", "llm"]])
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.9 · บันทึกลง Drive (และวิธีโหลดกลับมาใช้)

บันทึกเฉพาะ **LoRA adapter** (ไฟล์เล็ก หลักสิบ MB) ลง `MyDrive/taxi_qa/lora_taxi/` — ตัวโมเดลหลักไม่ต้องเก็บ (ดาวน์โหลดใหม่ได้)

**ครั้งหน้าจะใช้** (ไม่ต้องเทรนใหม่): รันส่วน 0–7 + 8.1 + 8.3 แล้วรัน
```python
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
tok = AutoTokenizer.from_pretrained(LLM_ID)
base = AutoModelForCausalLM.from_pretrained(LLM_ID, device_map="auto",
          quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
model = PeftModel.from_pretrained(base, str(DRIVE / "lora_taxi"))
```
แล้วต่อด้วย cell 8.8 (ฟังก์ชัน `llm_parse`) และ 8.10

In [ ]:
if RUN_LLM and IN_COLAB:
    model.save_pretrained(str(DRIVE / "lora_taxi"))
    tok.save_pretrained(str(DRIVE / "lora_taxi"))
    print("บันทึกแล้ว →", DRIVE / "lora_taxi")
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.10 · ถามด้วย LLM

`ask_llm()` ทำงานเหมือน `ask()` ส่วน 7 แต่ใช้ LLM เข้าใจคำถามแทนส่วน 3 + 5
- LLM บอกชื่อสถานที่เป็นข้อความ → ส่งต่อให้ `find_place()` แปลงเป็นรหัสอีกที (กันชื่อที่ LLM เขียนไม่ตรง)
- ทะเบียนและเวลา ใช้ตัวดึงของส่วน 3 ตามเดิม (แม่นกว่าและตรวจสอบได้)
- ค้นข้อมูลและตอบ ใช้ส่วน 6 + 7 เหมือนเดิม → **ตัวเลขมาจากข้อมูลจริงเสมอ**

In [ ]:
if RUN_LLM:
    def ask_llm(q):
        j = llm_parse(q)
        intent = j.get("intent")
        if intent not in set(T):
            return f"ไม่เข้าใจคำถาม (LLM ตอบ: {j.get('raw', j)})"
        if intent == "action_request":
            return "ระบบนี้ตอบคำถามได้อย่างเดียว ไม่สามารถสั่งการรถ (ตัดสัญญาณ/ล็อก/ดับเครื่อง) ได้"
        if intent == "other":
            return "ตอบได้เฉพาะคำถามเรื่องรถแท็กซี่ EV7"
        e = extract(q)                                             # ทะเบียน/เวลา จากตัวดึงส่วน 3
        place = find_place(j["place"]) if j.get("place") else e["place"]
        run = {"cars_in_area": lambda: x_cars_in_area(place, e["when"]),
               "car_status": lambda: x_car_status(e["plate"], e["when"]),
               "car_track": lambda: x_car_track(e["plate"], e["when"]),
               "out_of_zone": lambda: x_out_of_zone(e["when"]),
               "fleet_stats": lambda: x_fleet_stats(e["when"]),
               "nearest_charger": lambda: x_nearest_charger(e["plate"], place, e["when"])}[intent]
        return answer(intent, run())

    for q in ["ตอนนี้รถคันไหนอยู่ภูเกตบ้าง", f"{demo_plate} อยู่ไหน", "ช่วยดูหน่อยว่ารถที่อยู่โคราชตอนนี้มีกี่คัน"]:
        print("Q:", q); print("A:", ask_llm(q)); print()
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.11 · ลองถามเอง — ดูว่าระบบเข้าใจคำถามยังไง

`ลองถาม("คำถาม", ...)` แสดงทีละคำถาม:
1. **ค่าที่ดึงได้ (slot)** — ทะเบียน · สถานที่ (พร้อมคะแนนความเหมือน) · เวลา
2. **วิธีส่วน 5 (ตัวจัดประเภท)** — intent ที่ทาย + ความมั่นใจ + intent อันดับ 2–3 + คำตอบ
3. **วิธีส่วน 8 (LLM)** — JSON ที่ LLM เขียน + คำตอบ (แสดงเฉพาะเมื่อเทรน LLM แล้ว)

ใช้ได้ทั้งตอน `RUN_LLM = False` (เห็นแค่ข้อ 1–2) และ `True`

**ถามแบบพิมพ์สด:** รัน `ถามสด()` แล้วพิมพ์คำถามในช่องที่ขึ้นมา · กด Enter ว่าง ๆ เพื่อจบ

In [ ]:
def ลองถาม(*questions):
    llm_ready = "llm_parse" in globals()
    for q in questions:
        print("═" * 90); print("Q:", q)
        e = extract(q)
        pl = e["place"]
        place_txt = f"{pl['label']} ({pl['score']})" if pl else None
        print(f"  slot  ทะเบียน={e['plate']} · สถานที่={place_txt} · เวลา={e['when'][2]}")
        p = clf.predict_proba(enc.encode([mask(q)], normalize_embeddings=True))[0]
        top = np.argsort(p)[::-1][:3]
        print("  [ส่วน 5] intent:", " · ".join(f"{clf.classes_[k]} {p[k]:.2f}" for k in top))
        print("  [ส่วน 5] ตอบ:", ask(q))
        if llm_ready:
            print("  [LLM]  JSON:", llm_parse(q))
            print("  [LLM]  ตอบ:", ask_llm(q))
        else:
            print("  [LLM]  — ยังไม่ได้เทรน (RUN_LLM = False)")

def ถามสด():
    while True:
        q = input("ถาม (Enter ว่าง = จบ): ").strip()
        if not q:
            break
        ลองถาม(q)

ลองถาม("ตอนนี้รถคันไหนอยู่ภูเกตบ้าง",
       f"{demo_plate} อยู่ไหน",
       "รถที่อยู่ชลบุรีมีกี่คัน",
       f"เมื่อวาน {demo_plate} ไปไหนมาบ้าง",
       "ใกล้ๆหาดใหญ่มีที่ชาร์จไหม",
       "อยากรู้ว่ารถคันไหนวิ่งไปต่างจังหวัดบ้างวันนี้")

## 9 · ขั้นต่อไป — จาก notebook สู่การใช้งานจริง

### 9.1 เก็บคำถามจริงจากผู้ใช้ (สำคัญที่สุด)
ตอนนี้โมเดลเรียนจากประโยคที่เราแต่งเอง · ผู้ใช้จริงพิมพ์ไม่เหมือนที่เราคิดเสมอ
1. ให้ผู้ใช้ทดลองถาม (หรือเปิด Google Form ให้พิมพ์ "อยากถามอะไรระบบนี้") — เป้าหมายแรก **50–100 ข้อ**
2. บันทึกเป็นตาราง 2 คอลัมน์: `q` (คำถาม) · `intent` (คำตอบที่ถูก 1 ใน 8 แบบ) เช่น

   | q | intent |
   |---|---|
   | ภูเก็ตตอนนี้มีรถเรามั้ย | cars_in_area |
   | EV-0042 ตอนนี้ไปถึงไหนแล้ว | car_status |

3. **ข้อที่ไม่เข้า 8 แบบ** → จดแยกไว้ ถ้าเจอซ้ำบ่อยค่อยเพิ่ม intent ใหม่ (ต้องเขียน executor ส่วน 6 เพิ่มด้วย)

### 9.2 วงจรปรับปรุง (ทำซ้ำทุกครั้งที่มีคำถามใหม่)
```
คำถามจริง ─► เพิ่มเป็นชุดทดสอบ (HAND) ─► รันส่วน 5 ดูข้อที่ผิด
     ▲                                          │
     └── เทรนใหม่ ◄── เพิ่ม template ที่คล้ายข้อที่ผิด (ส่วน 4) ◄┘
```
อย่าเอาคำถามชุดทดสอบไปใส่ในชุดเทรนตรง ๆ — คะแนนจะสูงเกินจริง

### 9.3 เลือกว่าจะใช้วิธีไหนในระบบจริง
| ถ้า | ใช้ |
|---|---|
| คำถามส่วนใหญ่อยู่ใน 8 แบบ และส่วน 5 ถูก ≥ 90% | **ตัวจัดประเภทส่วน 5** — เร็ว ไม่ต้องใช้ GPU ข้อมูลไม่ออกนอกองค์กร |
| คำถามซับซ้อนเยอะ และ LLM (ส่วน 8) ถูกกว่าชัดเจน | **LLM รันเอง** — ต้องมีเครื่อง GPU ถาวร (Colab ฟรีใช้ทำ production ไม่ได้ ตัดการเชื่อมต่อบ่อย) |
| ไม่อยากดูแลเครื่อง GPU และ DPO อนุมัติให้ส่งข้อมูลออก | **API** (ดูโน้ต Q&A Assistant Design) |

### 9.4 ทำเป็นบริการให้หน้าแผนที่เรียกใช้
1. ย้ายโค้ดส่วน 1–7 เป็นไฟล์ `.py` บนเครื่องที่ข้อมูลอยู่ (ไม่ใช่ Colab)
2. เปิดเป็น API เล็ก ๆ ด้วย FastAPI — ตัวอย่างโครง:
   ```python
   from fastapi import FastAPI
   app = FastAPI()

   @app.get("/ask")
   def ask_api(q: str):
       return {"answer": ask(q)}
   ```
   รันด้วย `uvicorn app:app` → เรียก `http://<server>:8000/ask?q=...`
3. ให้ API คืน **รายการ car_id** ด้วย → หน้าแผนที่เอาไปไฮไลต์รถ และแปลงรหัสเป็นทะเบียนจริงฝั่งเว็บ (ทะเบียนจริงไม่ต้องผ่านโมเดล)
4. ต่อข้อมูลจริงแทนไฟล์ parquet เมื่อทำ realtime (Phase 6 ของแผนโปรเจกต์)

### 9.5 ความปลอดภัย
- ระบบนี้ **อ่านอย่างเดียว** · ไม่มีคำสั่งใดที่ควบคุมรถได้ (คำขอแบบนั้นถูกจัดเป็น `action_request` และปฏิเสธ)
- ล้าง output ก่อนส่ง notebook ต่อ (**Edit → Clear all outputs**)
- ใช้กับข้อมูลจริง (ทะเบียนจริง) เฉพาะบนเครื่ององค์กร · Colab ใช้กับชุดที่แทนทะเบียนแล้วเท่านั้น